In [ ]:
def read_file(fp):
    rows = []
    with open(fp, 'rb') as f:
        lines = f.readlines()
    
    for i, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(orjson.loads(line))
        except orjson.JSONDecodeError:
            if i < len(lines) - 1:  # only warn if it's NOT the last line
                print(f"ERROR mid-file {fp} line {i}")
            # silently skip truncated last line
    
    return rows

CSV_GLOB = "/scratch1/eibl/data/covid19_twitter/raw/*/*.json"
fps = sorted(glob.glob(CSV_GLOB))[:100]

with Pool() as pool:
    results = pool.map(read_file, fps)

df = pd.concat([pd.DataFrame(r) for r in results if r], ignore_index=True)
print(df.shape)

ERROR mid-file /scratch1/eibl/data/covid19_twitter/raw/coronavirus-2020-02-01/coronavirus-raw-2020-02-01-16.json line 81
ERROR mid-file /scratch1/eibl/data/covid19_twitter/raw/coronavirus-2020-02-01/coronavirus-raw-2020-02-01-16.json line 293
ERROR mid-file /scratch1/eibl/data/covid19_twitter/raw/coronavirus-2020-02-01/coronavirus-raw-2020-02-01-16.json line 498
ERROR mid-file /scratch1/eibl/data/covid19_twitter/raw/coronavirus-2020-02-01/coronavirus-raw-2020-02-01-16.json line 713
ERROR mid-file /scratch1/eibl/data/covid19_twitter/raw/coronavirus-2020-02-01/coronavirus-raw-2020-02-01-16.json line 928
ERROR mid-file /scratch1/eibl/data/covid19_twitter/raw/coronavirus-2020-02-01/coronavirus-raw-2020-02-01-16.json line 1137
ERROR mid-file /scratch1/eibl/data/covid19_twitter/raw/coronavirus-2020-02-01/coronavirus-raw-2020-02-01-16.json line 1344
ERROR mid-file /scratch1/eibl/data/covid19_twitter/raw/coronavirus-2020-02-01/coronavirus-raw-2020-02-01-16.json line 1547
ERROR mid-file /scratc

In [ ]:
df['tweet_hashtags'] = df.entities.str['hashtags'].apply(lambda x: [y['text'] for y in x])
df['tweet_user_mentions'] = df.entities.str['user_mentions'].apply(lambda x: [y['text'] for y in x])

In [ ]:
df['user_id'] = df.user.str['id']

In [ ]:
df.entities.apply(lambda x: x.keys())

In [ ]:
df['bio'] = df.user.str['description']

In [ ]:
df.bio.sample(20).values

import pandas as pd

maga = {'maga','kag','wwg1wga','trump2020','americafirst','kag2020','2a','nra','1a'}
hk   = {'standwithhongkong','fightforfreedom','followbackhongkong','standwithHongKong'}
left = {'resist'}
kpop = {'윤지성'}

def assign_label(hashtags: list[str]) -> str | None:
    tags = {h.lower().lstrip('#') for h in hashtags}
    votes = {
        'maga': len(tags & maga),
        'hk':   len(tags & hk),
        'left': len(tags & left),
        'kpop': len(tags & kpop),
    }
    # best = max(votes, key=votes.get)
    # return best if votes[best] > 0 else np.nan  # None = unlabeled
    return votes

df['bio'] = df.bio.replace({"": np.nan})
df['bio'] = df.bio.fillna("")
df['bio_hashtags'] = df.bio.str.findall("#\w+")

In [ ]:
pd.DataFrame(df.bio_hashtags.apply(assign_label).tolist()).gt(1).astype(int).value_counts(normalize=True)

In [46]:
df['urls'] = df[df['urls_raw'].str.len().gt(0)].urls.apply(lambda x: [xx['expanded_url'] for xx in x])

In [390]:
rep_media_pat = "|".join(REP_MEDIA_OUTLETS)
dem_media_pat = "|".join(DEM_MEDIA_OUTLETS)

df['rep_domains'] = df.urls.explode().str.findall(rep_media_pat).explode().dropna().groupby(level=0).agg(list)
user2rep_dom = df.explode('rep_domains').dropna(subset=['rep_domains']).groupby('userid')['rep_domains'].agg(list) 

df['has_rep_dom'] = df.rep_domains.str.len()
user2rep_dom_tweet_count = df.groupby('userid').has_rep_dom.sum()

df['dem_domains'] = df.urls.explode().str.findall(dem_media_pat).explode().dropna().groupby(level=0).agg(list)
user2dem_dom = df.explode('dem_domains').dropna(subset=['dem_domains']).groupby('userid')['dem_domains'].agg(list)

df['has_dem_dom'] = df.dem_domains.str.len()
user2dem_dom_tweet_count = df.groupby('userid').has_dem_dom.sum()

In [423]:
rep_hash_pat = r"\#(?:" + "|".join(REP_HASHTAGS) + r")"
dem_hash_pat = r"\#(?:" + "|".join(DEM_HASHTAGS) + r")"

# g = df.drop_duplicates(['userid','description']).copy()
df['rep_hashs'] = df.description.str.findall(rep_hash_pat)
df['rep_hashs_len'] = df['rep_hashs'].str.len()

user2rep_hash = (
    df.sort_values('rep_hashs_len')
        .drop_duplicates(['userid','description'], keep='first')
        .groupby('userid')
        .agg({'rep_hashs': 'sum'})
)
user2rep_hash['rep_hashs'] = user2rep_hash['rep_hashs'].apply(set)
user2rep_hash['rep_hashs_len'] = user2rep_hash.rep_hashs.str.len()

df['dem_hashs'] = df.description.str.findall(dem_hash_pat)
df['dem_hashs_len'] = df['dem_hashs'].str.len()

user2dem_hash = (
    df.sort_values('dem_hashs_len')
        .drop_duplicates(['userid','description'], keep='first')
        .groupby('userid')
        .agg({'dem_hashs': 'sum'})
)
user2dem_hash['dem_hashs'] = user2dem_hash['dem_hashs'].apply(set)
user2dem_hash['dem_hashs_len'] = user2dem_hash.dem_hashs.str.len()

In [431]:
user2pol = user2rep_hash.join(user2dem_hash)
user2pol['user2rep_dom_tweet_count'] = user2rep_dom_tweet_count
user2pol['user2dem_dom_tweet_count'] = user2dem_dom_tweet_count

In [ ]:
is_rep = (
    (user2pol.user2rep_dom_tweet_count.gt(0) | user2pol.rep_hashs_len.gt(0))
    # (counts.rep_dom_tweet_count.gt(0) & counts.rep_hash_count.gt(0))
)

is_dem = (
    (user2pol.user2dem_dom_tweet_count.gt(0) | user2pol.dem_hashs_len.gt(0))
    # (counts.dem_dom_tweet_count.gt(0) & counts.dem_hash_count.gt(0))
)

print((is_rep.sum(), is_dem.sum()), (is_rep.sum() + is_dem.sum()))

user2pol['is_rep'] = is_rep
user2pol['is_dem'] = is_dem
user2pol[['is_rep', 'is_dem']].sum(1).gt(1).mean(), user2pol[['is_rep', 'is_dem']].sum(1).gt(1).sum()

(6631, 28100) 34731


(0.008791281233887163, 3734)

In [ ]:
df['user2is_rep'] = df.userid.map(user2pol['is_rep'])
df['user2is_dem'] = df.userid.map(user2pol['is_dem'])
df['user2rep_dom_tweet_count'] = df.userid.map(user2pol.user2rep_dom_tweet_count)
df['user2dem_dom_tweet_count'] = df.userid.map(user2pol.user2dem_dom_tweet_count)
df['user2rep_hash_count'] = df.userid.map(user2pol.rep_hashs_len)
df['user2dem_hash_count'] = df.userid.map(user2pol.dem_hashs_len)

In [ ]:
user2pol = user2pol[~(user2pol.is_rep & user2pol.is_dem)]

,rep_hashs,rep_hashs_len,dem_hashs,dem_hashs_len,user2rep_dom_tweet_count,user2dem_dom_tweet_count,is_rep,is_dem
userid,,,,,,,,
5405752,{},0,{},0,1.0,0.0,True,True
5693842,{},0,{},0,1.0,0.0,True,True
6021962,{},0,{},0,1.0,0.0,True,True
6124022,{},0,{},0,1.0,0.0,True,True
7871422,{},0,{},0,1.0,0.0,True,True
...,...,...,...,...,...,...,...,...
1582450373201170432,{},0,{},0,1.0,0.0,True,True
1582697195354161152,{},0,{},0,1.0,0.0,True,True
1582926521303707648,{},0,{},0,1.0,0.0,True,True
